In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/maicnhd/dap391/sample_submission.csv
/kaggle/input/datasets/maicnhd/dap391/train_prompts.csv
/kaggle/input/datasets/maicnhd/dap391/test_essays.csv
/kaggle/input/datasets/maicnhd/dap391/train_essays.csv
/kaggle/input/datasets/thedrcat/daigt-v2-train-dataset/train_v2_drcat_02.csv


**DATA UNDERSTANDING**

In [2]:
# ============================================
# BƯỚC 1: HIỂU DỮ LIỆU (Data Understanding)
# ============================================
import pandas as pd
import numpy as np

train_df = pd.read_csv('/kaggle/input/datasets/thedrcat/daigt-v2-train-dataset/train_v2_drcat_02.csv')

print("Shape:", train_df.shape)
print("\nColumns:", train_df.columns.tolist())
print("\nData types:\n", train_df.dtypes)
print("\nMemory usage:\n", train_df.memory_usage(deep=True))
train_df.head()

Shape: (44868, 5)

Columns: ['text', 'label', 'prompt_name', 'source', 'RDizzl3_seven']

Data types:
 text             object
label             int64
prompt_name      object
source           object
RDizzl3_seven      bool
dtype: object

Memory usage:
 Index                  132
text             107185331
label               358944
prompt_name        3208059
source             2883562
RDizzl3_seven        44868
dtype: int64


,text,label,prompt_name,source,RDizzl3_seven
0,Phones\n\nModern humans today are always on th...,0,Phones and driving,persuade_corpus,False
1,This essay will explain if drivers should or s...,0,Phones and driving,persuade_corpus,False
2,Driving while the use of cellular devices\n\nT...,0,Phones and driving,persuade_corpus,False
3,Phones & Driving\n\nDrivers should not be able...,0,Phones and driving,persuade_corpus,False
4,Cell Phone Operation While Driving\n\nThe abil...,0,Phones and driving,persuade_corpus,False


**EDA**


In [3]:
# ============================================
# BƯỚC 2: KHÁM PHÁ DỮ LIỆU (EDA)
# ============================================
print("=== Phân bố nhãn ===")
print(train_df['label'].value_counts())
print(train_df['label'].value_counts(normalize=True) * 100)

print("\n=== Thống kê mô tả (numeric) ===")
print(train_df.describe())

print("\n=== Nếu có cột source (model sinh AI) ===")
if 'source' in train_df.columns:
    print(train_df['source'].value_counts())

=== Phân bố nhãn ===
label
0    27371
1    17497
Name: count, dtype: int64
label
0    61.003388
1    38.996612
Name: proportion, dtype: float64

=== Thống kê mô tả (numeric) ===
              label
count  44868.000000
mean       0.389966
std        0.487748
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000

=== Nếu có cột source (model sinh AI) ===
source
persuade_corpus                       25996
mistral7binstruct_v2                   2421
chat_gpt_moth                          2421
llama2_chat                            2421
mistral7binstruct_v1                   2421
kingki19_palm                          1384
train_essays                           1378
llama_70b_v1                           1172
falcon_180b_v1                         1055
darragh_claude_v6                      1000
darragh_claude_v7                      1000
radek_500                               500
NousResearch/Llama-2-7b-chat-hf         400
mistralai/Mistral-7B

In [4]:
# ============================================
# BƯỚC 3: XỬ LÝ MISSING DATA
# ============================================
print("=== Số lượng missing values theo cột ===")
print(train_df.isnull().sum())
print("\n=== Tỷ lệ missing (%) ===")
print(train_df.isnull().mean() * 100)

# Với dữ liệu text: không nên impute (điền giá trị) cho cột text bị thiếu,
# vì không thể "đoán" nội dung văn bản — chỉ nên loại bỏ dòng thiếu.
missing_text = train_df['text'].isnull().sum()
missing_label = train_df['label'].isnull().sum()
print(f"\nDòng thiếu text: {missing_text}, dòng thiếu label: {missing_label}")

train_df = train_df.dropna(subset=['text', 'label']).reset_index(drop=True)
print("Shape sau khi loại bỏ missing:", train_df.shape)

=== Số lượng missing values theo cột ===
text             0
label            0
prompt_name      0
source           0
RDizzl3_seven    0
dtype: int64

=== Tỷ lệ missing (%) ===
text             0.0
label            0.0
prompt_name      0.0
source           0.0
RDizzl3_seven    0.0
dtype: float64

Dòng thiếu text: 0, dòng thiếu label: 0
Shape sau khi loại bỏ missing: (44868, 5)


word_count (số từ trong văn bản) đóng 2 vai trò khác nhau trong đồ án của bạn — đây là điểm quan trọng cần phân biệt rõ:

Vai trò 1 — Công cụ làm sạch dữ liệu (đã dùng ở Bước 3b, 4)
Ở các bước Data Cleaning, word_count chỉ được dùng như một "cây thước đo" tạm thời để:

Lọc bỏ những dòng quá ngắn, không có ý nghĩa (< 20 từ) — vì 1 câu vài từ không đủ để phân biệt human hay AI viết.
Phát hiện outlier cực đoan (Bước 4) — dòng có độ dài bất thường có thể do lỗi crawl dữ liệu.

Ở vai trò này, nó giống như bạn cân sản phẩm để loại bỏ hàng lỗi trước khi đóng gói — dùng xong mục đích lọc là xong, không nhất thiết phải giữ lại.

Vai trò 2 — Đặc trưng (feature) đưa vào model phân loại (sẽ dùng ở Bước 5)
Đây mới là vai trò quan trọng hơn cho bài toán chính của bạn. Từ biểu đồ boxplot bạn vẽ trước đó, word_count cho thấy:

Human viết có độ dài dao động rộng (ngắn/dài thất thường).
AI viết có xu hướng độ dài đều đặn hơn, ít cực đoan.

→ Sự khác biệt này, dù không quá mạnh một mình, vẫn là 1 tín hiệu (signal) giúp model học được: "nếu độ dài rơi vào khoảng X, khả năng cao là nhóm Y". Khi đưa vào model (Logistic Regression, Random Forest...), word_count sẽ là 1 cột input cùng với các đặc trưng khác (TF-IDF, avg_sentence_length, perplexity...) để model tổng hợp và đưa ra quyết định phân loại.

Ví dụ dễ hình dung: giống như bạn đoán ai là "dân văn phòng" hay "sinh viên" — chỉ nhìn chiều cao thì không chắc chắn được (vì cả 2 nhóm đều có người cao/thấp), nhưng nếu kết hợp thêm cách ăn mặc, giờ giấc, đồ mang theo... thì đoán chính xác hơn nhiều. word_count giống "chiều cao" — một mảnh ghép nhỏ, không quyết định, nhưng góp phần vào bức tranh tổng thể mà model học được.

In [5]:
# ============================================
# BƯỚC 3b: XỬ LÝ TEXT RỖNG / QUÁ NGẮN
# ============================================
# Loại bỏ text rỗng, chỉ có khoảng trắng, hoặc quá ngắn để có ý nghĩa (< 20 từ)
train_df['text'] = train_df['text'].astype(str).str.strip()
train_df = train_df[train_df['text'].str.len() > 0]

train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))
too_short = train_df[train_df['word_count'] < 20]
print(f"Số dòng quá ngắn (<20 từ): {len(too_short)}")

train_df = train_df[train_df['word_count'] >= 20].reset_index(drop=True)
print("Shape sau khi loại text quá ngắn:", train_df.shape)

Số dòng quá ngắn (<20 từ): 2
Shape sau khi loại text quá ngắn: (44866, 6)


In [6]:
print("Shape hiện tại:", train_df.shape)
print("Phân bố nhãn hiện tại:")
print(train_df['label'].value_counts())

Shape hiện tại: (44866, 6)
Phân bố nhãn hiện tại:
label
0    27371
1    17495
Name: count, dtype: int64


So sánh với lần lọc global cũ (loại 339 dòng) — vì sao lần này chỉ loại 36 dòng:
Đây là điểm quan trọng cần hiểu: lọc group-wise loại ít dòng hơn hẳn (36 so với 339) vì mỗi nhóm được đánh giá theo đúng phân bố của chính nó. Ví dụ label 1 (AI) vốn có xu hướng ngắn và đều hơn, ngưỡng trên tự nhiên của nó là 722 — nếu dùng ngưỡng global (982, tính chung cả 2 nhóm) sẽ bỏ sót nhiều outlier thật của riêng label 1 (những bài AI dài bất thường so với các bài AI khác, dù chưa chạm ngưỡng 982). Ngược lại, group-wise sẽ bắt đúng outlier thật của từng nhóm — đây là lý do phương pháp này chính xác hơn về mặt thống kê, dù số dòng bị loại ít hơn.

In [7]:
# ============================================
# BƯỚC 4 (DUY NHẤT — group-wise, không lọc global trước đó nữa)
# ============================================
def remove_outliers_iqr(df, col, group_col, k=3):
    df_list = []
    for label_value, group in df.groupby(group_col):
        Q1 = group[col].quantile(0.25)
        Q3 = group[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = max(0, Q1 - k * IQR)
        upper = Q3 + k * IQR
        filtered = group[(group[col] >= lower) & (group[col] <= upper)]
        print(f"Label {label_value}: upper={upper:.0f} → loại {len(group)-len(filtered)}/{len(group)} dòng")
        df_list.append(filtered)
    return pd.concat(df_list).reset_index(drop=True)

train_df = remove_outliers_iqr(train_df, 'word_count', 'label', k=3)
print("Shape sau khi lọc outlier (1 lần duy nhất, group-wise):", train_df.shape)

Label 0: upper=1254 → loại 16/27371 dòng
Label 1: upper=722 → loại 20/17495 dòng
Shape sau khi lọc outlier (1 lần duy nhất, group-wise): (44830, 6)


In [8]:
# ============================================
# BƯỚC 5: CHUẨN HÓA & TRANSFORM DATA (làm sạch text)
# ============================================
import re

def clean_text(text):
    text = text.lower()                                  # chuyển chữ thường
    text = re.sub(r'http\S+|www\S+', '', text)            # xóa URL
    text = re.sub(r'\S+@\S+', '', text)                   # xóa email
    text = re.sub(r'<.*?>', '', text)                      # xóa HTML tag nếu có
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?\']', ' ', text)  # giữ lại chữ, số, dấu câu cơ bản
    text = re.sub(r'\s+', ' ', text).strip()               # xóa khoảng trắng thừa
    return text

train_df['text_clean'] = train_df['text'].apply(clean_text)

# Cập nhật lại word_count sau khi làm sạch
train_df['word_count'] = train_df['text_clean'].apply(lambda x: len(x.split()))

train_df[['text', 'text_clean']].head(10)

,text,text_clean
0,Phones\n\nModern humans today are always on th...,phones modern humans today are always on their...
1,This essay will explain if drivers should or s...,this essay will explain if drivers should or s...
2,Driving while the use of cellular devices\n\nT...,driving while the use of cellular devices toda...
3,Phones & Driving\n\nDrivers should not be able...,phones driving drivers should not be able to u...
4,Cell Phone Operation While Driving\n\nThe abil...,cell phone operation while driving the ability...
5,Cell phone use should not be legal while drivi...,cell phone use should not be legal while drivi...
6,Phones and Driving\n\nDriving is a good way to...,phones and driving driving is a good way to ge...
7,PHONES AND DRIVING\n\nIn this world in which w...,phones and driving in this world in which we l...
8,People are debating whether if drivers should ...,people are debating whether if drivers should ...
9,Texting and driving\n\nOver half of drivers in...,texting and driving over half of drivers in to...


text_clean là cột bạn đã tạo ra ở Bước 5 (Chuẩn hóa & Transform Data) — tin nhắn trước đó mình gửi code clean_text() cho bạn, và nó đã chạy đúng trước khi bạn sang Bước 6. Cột này có 2 tác dụng chính:

1. Tác dụng bạn vừa thấy — dùng để phát hiện trùng lặp chính xác hơn (Bước 6 hiện tại)

Nhìn vào output: duplicated() (trùng hoàn toàn) chỉ ra 1 dòng, nhưng duplicated(subset=['text_clean']) ra tới 12 dòng. Sự chênh lệch này rất có ý nghĩa:

Có thể có 2 bài viết nội dung giống hệt nhau nhưng khác nhau ở chữ hoa/thường, dấu câu, hoặc khoảng trắng thừa (ví dụ: "The Car." vs "the car") → nếu so trên cột text gốc, pandas coi là 2 dòng khác nhau (vì khác ký tự), nhưng thực chất là cùng 1 nội dung.
So trên text_clean (đã lowercase, xóa ký tự thừa) sẽ bắt được các trùng lặp "ẩn" này — giúp dataset sạch hơn, tránh model học đi học lại cùng 1 mẫu, gây overfitting nhẹ.

→ Đây là lý do bạn nên luôn so trùng lặp trên bản đã chuẩn hóa, không phải bản gốc — và bạn đã làm đúng.

2. Tác dụng quan trọng hơn — sẽ dùng ở Bước 5 xây model (TF-IDF)

Đây là mục đích chính khi tạo cột này ngay từ đầu: text_clean sẽ là input trực tiếp cho bước vector hóa văn bản (TF-IDF hoặc CountVectorizer) trước khi đưa vào 5 model phân loại. Lý do phải dùng bản đã làm sạch thay vì text gốc:

Nếu để nguyên chữ hoa/thường, model TF-IDF sẽ coi "The" và "the" là 2 từ khác nhau → làm phình to số chiều (dimension) vô ích, giảm hiệu quả.
Nếu còn URL, email, HTML tag lẫn trong text → đây là nhiễu, không mang thông tin phân loại human/AI, chỉ làm rối mô hình.
Dấu câu cơ bản (., ,, !, ?) mình cố tình giữ lại (như đã giải thích ở tin trước) vì đó là tín hiệu quan trọng để phân biệt AI vs Human.

In [9]:
# ============================================
# BƯỚC 6: XỬ LÝ DỮ LIỆU TRÙNG LẶP
# ============================================
print("Số dòng trùng lặp hoàn toàn:", train_df.duplicated().sum())
print("Số dòng trùng lặp theo text_clean:", train_df.duplicated(subset=['text_clean']).sum())

before = train_df.shape[0]
train_df = train_df.drop_duplicates(subset=['text_clean']).reset_index(drop=True)
after = train_df.shape[0]

print(f"Đã xóa {before - after} dòng trùng lặp. Shape hiện tại: {train_df.shape}")

Số dòng trùng lặp hoàn toàn: 1
Số dòng trùng lặp theo text_clean: 12
Đã xóa 12 dòng trùng lặp. Shape hiện tại: (44818, 7)


In [10]:
print(train_df.columns.tolist())

['text', 'label', 'prompt_name', 'source', 'RDizzl3_seven', 'word_count', 'text_clean']


In [11]:
# ============================================
# BƯỚC 7: KIỂM TRA VÀ VALIDATE KẾT QUẢ
# ============================================
# Kiểm tra lại toàn bộ sau khi clean: không còn missing, không còn trùng, nhãn vẫn đủ 2 lớp
assert train_df['text_clean'].isnull().sum() == 0, "Vẫn còn missing text!"
assert train_df.duplicated(subset=['text_clean']).sum() == 0, "Vẫn còn trùng lặp!"
assert train_df['label'].nunique() == 2, "Thiếu 1 trong 2 nhãn sau khi lọc!"

print("=== Phân bố nhãn SAU khi clean ===")
print(train_df['label'].value_counts())
print(train_df['label'].value_counts(normalize=True) * 100)

print("\n=== Thống kê word_count SAU khi clean ===")
print(train_df.groupby('label')['word_count'].describe())

print("\n✅ Dữ liệu đã qua kiểm tra, sẵn sàng lưu.")

=== Phân bố nhãn SAU khi clean ===
label
0    27343
1    17475
Name: count, dtype: int64
label
0    61.00897
1    38.99103
Name: proportion, dtype: float64

=== Thống kê word_count SAU khi clean ===
         count        mean         std    min    25%    50%    75%     max
label                                                                     
0      27343.0  419.561862  189.145485  143.0  275.0  385.0  521.0  1255.0
1      17475.0  330.789585   94.000390   42.0  276.0  328.0  388.0   728.0

✅ Dữ liệu đã qua kiểm tra, sẵn sàng lưu.


In [12]:
# ============================================
# BƯỚC 8: LƯU DỮ LIỆU SẠCH
# ============================================
output_path = '/kaggle/working/train_cleaned.csv'
train_df.to_csv(output_path, index=False)

print(f"Đã lưu dữ liệu sạch tại: {output_path}")
print(f"Shape cuối cùng: {train_df.shape}")
print(f"Các cột: {train_df.columns.tolist()}")

Đã lưu dữ liệu sạch tại: /kaggle/working/train_cleaned.csv
Shape cuối cùng: (44818, 7)
Các cột: ['text', 'label', 'prompt_name', 'source', 'RDizzl3_seven', 'word_count', 'text_clean']
